
# Pipeline overview 

- Load **X** (genes) and **Y** (proteins); pick **one protein** to predict.
- Take a **tiny subset** of samples/genes to keep it fast.
- Do **leave-one-out (LOO)**: for each sample, hold it out as test.
- On the remaining training set, run **inner K-fold CV** to pick XGBoost hyperparams.
- Within each inner fold, do **feature selection**:
  - Compute **p-values** per gene (default: **Pearson**).
  - Keep the **top N%** with lowest p-values (default: **5%**).
- Train XGBoost on the selected features for each param combo; pick the one with **lowest MSE**.
- **Recompute feature selection** on full train with the chosen threshold; **refit** XGBoost and **predict** the held-out sample.
- Collect a row per test sample: **true value**, **predicted value**, **chosen params**, and (if available) **feature importances**.
- Finally, show a quick **RMSE** over the LOO predictions.



In [1]:

# --- Configure paths ---
import os, sys, warnings
warnings.filterwarnings("ignore")

# Project layout expectation:
# <project_root>/
#   ├─ src/pipeline_functions.py
#   └─ data/*.joblib
project_root = os.path.abspath(os.getcwd())
src_dir = os.path.join(project_root, "src")
data_dir = os.path.join(project_root, "data")
os.makedirs(data_dir, exist_ok=True)

# Make sure we can import pipeline_functions.py
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("project_root:", project_root)
print("src_dir:", src_dir)
print("data_dir:", data_dir)


project_root: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb
src_dir: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\src
data_dir: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\pipeline_xgb\data


In [2]:

# --- Load or create tiny data ---
from joblib import load, dump
import pandas as pd
import numpy as np

X_path = os.path.join(data_dir, "transcriptomics_filtered.joblib")
Y_path = os.path.join(data_dir, "proteomics_matched.joblib")

def create_synthetic_joblibs(n_samples=40, n_genes=500, n_proteins=10, seed=123):
    rng = np.random.default_rng(seed)
    # Features
    X = pd.DataFrame(rng.normal(size=(n_samples, n_genes)).astype(np.float32),
                     index=[f"S{i:03d}" for i in range(n_samples)],
                     columns=[f"G{i:05d}" for i in range(n_genes)])
    X.index.name = "samples"; X.columns.name = "genes"
    # Targets as a noisy linear combo of a few genes
    Y = pd.DataFrame(index=X.index, columns=[f"P{i:04d}" for i in range(n_proteins)], dtype=np.float32)
    for j in range(n_proteins):
        coef = rng.normal(size=n_genes).astype(np.float32)
        mask = rng.choice([0,1], size=n_genes, p=[0.98, 0.02]).astype(bool)  # 2% sparse
        coef *= mask
        y = X.values @ coef + rng.normal(scale=0.5, size=n_samples)
        Y.iloc[:, j] = y.astype(np.float32)
    Y.index.name = "samples"; Y.columns.name = "proteins"
    dump(X, X_path); dump(Y, Y_path)
    print(f"Synthetic data saved ->\n  {X_path}\n  {Y_path}")
    return X, Y

if not (os.path.exists(X_path) and os.path.exists(Y_path)):
    print("Joblib matrices not found. Creating a small synthetic dataset...")
    X, Y = create_synthetic_joblibs()
else:
    X = load(X_path); Y = load(Y_path)

print("X shape:", X.shape, " | Y shape:", Y.shape)
print("X head:"); display(X.head(2))
print("Y head:"); display(Y.head(2))

# Basic alignment check
assert X.index.equals(Y.index), "Sample indices of X and Y must match!"


X shape: (369, 22614)  | Y shape: (369, 5113)
X head:


,A1BG,A1BG-AS1,A1CF,A2M,A2ML1,A4GALT,AAAS,AACS,AACSP1,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
MDAMB468_BREAST,2.689299,2.954196,0.014355,3.329124,0.443607,1.575312,5.311430,4.531069,0.056584,0.124328,...,4.139142,6.986752,2.192194,3.114367,4.250962,2.792855,3.250962,6.233811,3.512227,5.221490
SH4_SKIN,4.980025,3.583760,0.000000,8.405099,0.000000,0.111031,5.229972,4.095924,0.070389,0.014355,...,3.698218,6.113742,1.384050,2.659925,4.979111,0.014355,2.553361,6.644000,3.659925,4.249445


Y head:


Gene_Symbol,A2M,AAAS,AACS,AAGAB,AAK1,AAMDC,AAMP,AAR2,AARS,AARS2,...,ZNF787,ZNF800,ZNF830,ZNHIT6,ZNRD1,ZRANB2,ZW10,ZWILCH,ZYX,ZZZ3
MDAMB468_BREAST,-0.345562,-0.109405,0.370945,0.009639,0.005800,1.588762,-0.364119,-0.031192,-0.052310,-0.282176,...,-0.094039,-0.273452,-0.222986,0.110619,-0.220072,0.320284,-0.109837,-0.234132,-1.032888,-0.054057
SH4_SKIN,1.642186,-0.283335,0.162102,-0.057517,0.760639,0.119699,0.105279,-0.075316,0.399385,1.102265,...,-0.248390,0.531159,-0.452978,-0.386262,-0.047313,-0.499179,-0.208351,-0.163901,-0.361225,-0.371537


In [3]:

# --- Import your pipeline and set a tiny param grid (CPU-only) ---
from pipeline_functions import cv_pipeline_fixed_percentile
import xgboost as xgb

# pick a single protein and a tiny subset to keep it FAST
protein = Y.columns[0]
n_samples = min(10, len(X))      # use at most 10 samples
n_genes_subset = min(400, X.shape[1])  # at most 400 genes

X_small = X.iloc[:n_samples, :n_genes_subset].copy()
Y_small = Y.iloc[:n_samples, [0]].copy()  # keep only the first protein

# minimal param grid for a super-quick run (CPU)
param_grid = {
    "max_depth": [3],
    "learning_rate": [0.1],
    "n_estimators": [50],
    "subsample": [1.0],
    "colsample_bytree": [1.0],
    "tree_method": ["hist"],  # CPU; switch to 'gpu_hist' if you want to test GPU here
}

model = xgb.XGBRegressor(objective="reg:squarederror", n_jobs=-1, random_state=42, verbosity=0)
print("Testing protein:", protein)
print("Subset X:", X_small.shape, "Subset Y:", Y_small.shape)


Testing protein: A2M
Subset X: (10, 400) Subset Y: (10, 1)


In [4]:

# --- Run a tiny LOO over the subset ---
results_df = cv_pipeline_fixed_percentile(
    model=model,
    param_grid=param_grid,
    X=X_small,
    y=Y_small,
    protein_name=Y_small.columns[0],
    feature_selector="pearson",
    n_splits=3,          # small inner CV
    percentile=5.0       # keep top 5% features by lowest p-value
)

print("Results shape:", results_df.shape)
display(results_df.head())

# Quick RMSE over the subset
from sklearn.metrics import mean_squared_error
import numpy as np
mask = np.isfinite(results_df["predicted"].astype(float))
rmse = mean_squared_error(results_df.loc[mask, "true"], results_df.loc[mask, "predicted"], squared=False)
print(f"Subset RMSE: {rmse:.4f}")


Results shape: (10, 7)


,protein,LOO sample,true,predicted,params_dict,features_coef_dict,percentile
0,A2M,MDAMB468_BREAST,-0.345562,1.461154,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...","{'AAAS': 0.013963301666080952, 'ABCA2': 0.7346...",5.0
1,A2M,SH4_SKIN,1.642186,-0.217702,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...","{'AAMP': 0.002626970177516341, 'ABCA2': 0.0571...",5.0
2,A2M,AU565_BREAST,-1.129114,1.362478,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...","{'ABCG1': 0.007924174889922142, 'ABHD4': 0.003...",5.0
3,A2M,KMRC1_KIDNEY,1.493333,-1.037115,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...","{'A2M': 0.036780621856451035, 'AARS2': 0.06440...",5.0
4,A2M,CAL51_BREAST,0.274652,-0.235732,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...","{'AAAS': 0.003928466234356165, 'ABCA2': 0.1835...",5.0


Subset RMSE: 2.0161


In [5]:

# --- (Optional) Quick GPU sanity check ---
# Tries a single tiny fit with 'gpu_hist'. This does NOT use the full pipeline; just a probe.
import xgboost as xgb
from sklearn.model_selection import train_test_split

try:
    Xt = X_small.values
    yt = Y_small.iloc[:,0].values
    Xtr, Xte, ytr, yte = train_test_split(Xt, yt, test_size=0.2, random_state=42)

    gpu_model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=20,
        max_depth=3,
        learning_rate=0.1,
        tree_method="gpu_hist",  # requires GPU
        random_state=42,
        verbosity=0,
    )
    gpu_model.fit(Xtr, ytr)
    print("✅ GPU probe fit succeeded (tree_method='gpu_hist').")
    print("  Booster params:", gpu_model.get_xgb_params())
except Exception as e:
    print("⚠️ GPU probe failed. That's okay on CPU-only machines. Error:")
    print(e)


⚠️ GPU probe failed. That's okay on CPU-only machines. Error:
[17:55:15] C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\tree\updater_gpu_hist.cu:861: Exception in gpu_hist: [17:55:15] C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\tree\updater_gpu_hist.cu:867: Check failed: ctx_->Ordinal() >= 0 (-1 vs. 0) : Must have at least one device




## Next steps

- If this notebook runs without errors and prints a reasonable RMSE with a small result table, your wiring is good.
- To dry-run your Slurm job locally, set `tree_method="hist"` as above. On the HPC, use `--force_gpu` or `tree_method="gpu_hist"` and the GPU partition.
- When you're happy, launch your full run via the `submit_array_chunks.sh` script.
